# POI Ingestion - Bronze Layer

Queries the Overpass API (OpenStreetMap) for partner and competitor brand locations.

**Data Source:** Overpass API - queries OSM data for specific brand names across the US

**Why Overpass instead of full PBF download?**
- We only need ~50-200 branded POIs, not millions of generic OSM elements
- General POI counts (retail, food_drink, etc.) come from CARTO Marketplace at H3 level
- Overpass queries complete in seconds vs 13+ minutes for full PBF parsing

**Brand Configuration:** Loaded from `poi_config.yml` (single source of truth)

**Output Table:**
- `{catalog}.{bronze_schema}.raw_pois` - Branded POI data with tags

## Parameters

In [ ]:
import requests
import time
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import *
from collections import Counter

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("config_path", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
config_path = dbutils.widgets.get("config_path")

assert catalog and bronze_schema and config_path, "Missing required parameters: catalog, bronze_schema, config_path"

output_table = f"{catalog}.{bronze_schema}.raw_pois"

print(f"Catalog: {catalog}")
print(f"Schema: {bronze_schema}")
print(f"Config: {config_path}")
print(f"Output table: {output_table}")

## Load Brand Configuration

In [ ]:
# Load brand configuration from poi_config.yml (single source of truth)
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

brands_config = config.get('brands', {})
partner_brands = brands_config.get('partner_brands', [])
competitor_brands = brands_config.get('competitor_brands', [])

assert partner_brands, "No partner_brands defined in poi_config.yml"
assert competitor_brands, "No competitor_brands defined in poi_config.yml"

print(f"Partner brands ({len(partner_brands)}): {partner_brands}")
print(f"Competitor brands ({len(competitor_brands)}): {competitor_brands}")

# Build Overpass regex from brand names
# Use set of base names (deduplicate variants like "Domino's" and "Dominos")
# Replace apostrophes with .? and hyphens with . for flexible matching
all_brands = partner_brands + competitor_brands
base_names = set()
for brand in all_brands:
    base = brand.replace("'", "").replace("'", "").replace("-", " ").split()[0]
    base_names.add(brand)

# Build regex patterns - handle special characters for Overpass POSIX regex
regex_parts = []
seen = set()
for brand in all_brands:
    # Normalize: "Domino's" and "Dominos" → same pattern "Domino.?s"
    pattern = brand.replace("'", ".?").replace("'", ".?").replace("-", ".")
    if pattern.lower() not in seen:
        regex_parts.append(pattern)
        seen.add(pattern.lower())

brand_regex = "|".join(regex_parts)
print(f"\nOverpass regex: {brand_regex}")

## Query Overpass API

In [ ]:
# Overpass API endpoints (primary + fallback)
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

# US contiguous bounding box (south, west, north, east)
US_BBOX = "24.0,-125.0,50.0,-66.0"

# Build Overpass QL query
# Searches for nodes and ways with matching brand names across the US
# 'out center' returns centroid coordinates for ways (building polygons)
overpass_query = f"""
[out:json][timeout:120];
(
  node["name"~"{brand_regex}",i]({US_BBOX});
  way["name"~"{brand_regex}",i]({US_BBOX});
);
out center;
"""

print(f"Overpass query:\n{overpass_query}")

def query_overpass(query, endpoints=OVERPASS_ENDPOINTS, max_retries=3):
    """Execute Overpass query with retry and endpoint fallback."""
    for endpoint in endpoints:
        for attempt in range(max_retries):
            try:
                print(f"  Querying {endpoint} (attempt {attempt + 1})...")
                response = requests.post(
                    endpoint,
                    data={"data": query},
                    timeout=120,
                    headers={"User-Agent": "DatabricksGeospatialPipeline/1.0"}
                )
                response.raise_for_status()
                data = response.json()
                element_count = len(data.get("elements", []))
                print(f"  ✓ Received {element_count} elements")
                return data
            except requests.exceptions.Timeout:
                print(f"  Timeout on attempt {attempt + 1}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
            except requests.exceptions.HTTPError as e:
                if e.response and e.response.status_code == 429:
                    wait = 2 ** (attempt + 2)
                    print(f"  Rate limited, waiting {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  HTTP error: {e}")
                    break
            except Exception as e:
                print(f"  Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    break
        print(f"  Failed on {endpoint}, trying next...")
    raise RuntimeError("All Overpass endpoints failed after retries")

# Execute query
start_time = time.time()
result = query_overpass(overpass_query)
query_time = time.time() - start_time
print(f"\nQuery completed in {query_time:.1f}s")

In [ ]:
# Parse Overpass JSON response into POI list
# Output schema matches original raw_pois: osm_id, osm_type, latitude, longitude, tags
pois = []
skipped = 0

for element in result.get("elements", []):
    osm_type = element["type"]
    osm_id = str(element["id"])
    tags = element.get("tags", {})

    # Get coordinates: nodes have lat/lon directly, ways have center
    if osm_type == "node":
        lat = element.get("lat")
        lon = element.get("lon")
    elif osm_type == "way":
        center = element.get("center", {})
        lat = center.get("lat")
        lon = center.get("lon")
    else:
        skipped += 1
        continue

    if lat is not None and lon is not None:
        pois.append({
            "osm_id": osm_id,
            "osm_type": osm_type,
            "latitude": float(lat),
            "longitude": float(lon),
            "tags": tags
        })
    else:
        skipped += 1

print(f"Parsed {len(pois)} POIs ({skipped} skipped - missing coordinates)")

In [ ]:
# Extraction summary
node_count = sum(1 for p in pois if p["osm_type"] == "node")
way_count = sum(1 for p in pois if p["osm_type"] == "way")

print(f"{'='*60}")
print(f"POI EXTRACTION SUMMARY")
print(f"{'='*60}")
print(f"Total POIs: {len(pois)}")
print(f"  Nodes: {node_count}")
print(f"  Ways:  {way_count}")
print(f"  Query time: {query_time:.1f}s")

# Count by brand name
brand_counts = Counter()
for p in pois:
    name = p["tags"].get("name", "Unknown")
    brand_counts[name] += 1

print(f"\nPOIs by brand:")
for brand, count in brand_counts.most_common():
    print(f"  {brand}: {count}")

if len(pois) == 0:
    raise RuntimeError(
        "No POIs found. Check brand patterns in poi_config.yml "
        "and Overpass API connectivity."
    )

## Write to Bronze Table

In [ ]:
# Convert POIs to Spark DataFrame (same schema as original raw_pois)
schema = StructType([
    StructField("osm_id", StringType(), False),
    StructField("osm_type", StringType(), False),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("tags", MapType(StringType(), StringType()), True)
])

poi_df = spark.createDataFrame(pois, schema=schema)
poi_df = poi_df.withColumn("ingestion_timestamp", F.current_timestamp())

print(f"Created DataFrame with {poi_df.count()} rows")
display(poi_df.limit(10))

In [ ]:
# Write to Bronze table
(poi_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(output_table))

print(f"✓ Written {poi_df.count()} POIs to {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("POI INGESTION VALIDATION")
print("=" * 80)

summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_pois,
        COUNT(DISTINCT osm_id) as unique_pois,
        COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as pois_with_coords,
        COUNT(CASE WHEN tags['name'] IS NOT NULL THEN 1 END) as pois_with_name,
        COUNT(CASE WHEN tags['addr:street'] IS NOT NULL THEN 1 END) as pois_with_address
    FROM {output_table}
""")
display(summary)

# Show all extracted branded POIs
print("\nExtracted branded POIs:")
spark.sql(f"""
    SELECT 
        tags['name'] as name,
        COALESCE(tags['amenity'], tags['shop']) as category,
        osm_type,
        COUNT(*) as count
    FROM {output_table}
    WHERE tags['name'] IS NOT NULL
    GROUP BY tags['name'], COALESCE(tags['amenity'], tags['shop']), osm_type
    ORDER BY count DESC
""").show(50, truncate=False)

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)